In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [6]:
CSV_PATH = 'raw_car_dataset.csv'
df = pd.read_csv(CSV_PATH)

In [7]:
print("\n=== INITIAL HEAD ===")
print(df.head(10))


=== INITIAL HEAD ===
    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022
5  $1,500     211171.0    4.0          0       ??  2003
6  1500.0     222235.0    4.0          2    Rural  2004
7  $1,500     105068.0    5.0          1     City  2002
8  $2,291      90015.0    4.0          0    Rural  2007
9  1500.0     125976.0    2.0          0     City  2002


In [8]:
print("\n=== INITIAL INFO ===")
print(df.info())


=== INITIAL INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    object 
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    object 
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 6.9+ KB
None


In [9]:
print("\n=== INITIAL MISSING VALUES ===")
print(df.isnull().sum())


=== INITIAL MISSING VALUES ===
Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [10]:
print(df['Location'].value_counts(dropna=False))

Location
City      59
Suburb    45
Rural     21
Subrb      8
??         7
NaN        5
Name: count, dtype: int64


In [11]:
df["Price"] = df["Price"].replace(r"[\$,]", "", regex=True).astype(float)
print(df["Price"].head(10))

0    1500.0
1    4171.0
2    5331.0
3    1500.0
4    1500.0
5    1500.0
6    1500.0
7    1500.0
8    2291.0
9    1500.0
Name: Price, dtype: float64


In [12]:
print("Dtype:", df["Price"].dtype)
print("Skewness:",df["Price"].skew())

Dtype: float64
Skewness: 7.871113660850301


In [13]:
df["Location"] = df["Location"].replace({"Subrb": "Suburb", "??": pd.NA})
print("Location counts after type/unknown fix")
print(df["Location"].value_counts(dropna=False))

Location counts after type/unknown fix
Location
City      59
Suburb    53
Rural     21
<NA>       7
NaN        5
Name: count, dtype: int64


In [14]:
df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"]  = df["Doors"].fillna(df["Doors"].mode()[0])
df["Accidents"]  = df["Accidents"].fillna(df["Accidents"].mode()[0])
df["Location"]  = df["Location"].fillna(df["Location"].mode()[0])
print(df.isnull().sum())

Price          0
Odometer_km    0
Doors          0
Accidents      0
Location       0
Year           0
dtype: int64


In [15]:
before = df.shape
df = df.drop_duplicates()
after = df.shape
print(f"Dropped duplicates: {before} → {after}")

Dropped duplicates: (145, 6) → (140, 6)


In [18]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper

low_price, high_price = iqr_fun(df["Price"])
low_Odometer_km,  high_Odometer_km  = iqr_fun(df["Odometer_km"])

df["Price"]     = df["Price"].clip(lower=low_price, upper=high_price)
df["Odometer_km"] = df["Odometer_km"].clip(lower=low_Odometer_km,  upper=high_Odometer_km)


In [19]:
df = pd.get_dummies(df, columns=["Location"], drop_first=False, dtype="int")

In [24]:
df["CarAge"] = 2026 - df["Year"] 

df["Km_per_year"] = (
    df["Odometer_km"] /
    df["CarAge"].replace(0, 1)
)

df["Is_Urban"] = (
    df["Location_City"]
    .astype(int)
)

df["LogPrice"] = np.log(
    df["Price"] + 1
)

df[
    [
        "CarAge",
        "Km_per_year",
        "Is_Urban",
        "LogPrice"
    ]
].head()

,CarAge,Km_per_year,Is_Urban,LogPrice
0,28,4922.500000,1,7.313887
1,10,12854.800000,0,8.336151
2,12,8941.833333,0,8.581482
3,27,5253.259259,0,7.313887
4,4,32137.000000,1,7.313887


In [25]:
from sklearn.preprocessing import StandardScaler

dont_scale = {'Price', 'LogPrice'}
exclude = [c for c in df.columns if c.startswith('Location_')] + ['Is_Urban']
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in dont_scale and c not in exclude]

df[scale_cols] = StandardScaler().fit_transform(df[scale_cols])
df[scale_cols].head()

,Odometer_km,Doors,Accidents,Year,CarAge,km_per_year,Km_per_year
0,0.128390,0.254091,0.316968,-1.686714,1.686714,-0.615631,-0.615631
1,-0.044709,0.254091,-0.820867,0.794617,-0.794617,0.070446,0.070446
2,-0.440923,0.254091,-0.820867,0.518913,-0.518913,-0.267993,-0.267993
3,0.203135,0.254091,0.316968,-1.548862,1.548862,-0.587024,-0.587024
4,-0.044709,-0.931668,-0.820867,1.621727,-1.621727,1.738196,1.738196


In [26]:
assert df["Price"].dtype == float

assert "LogPrice" in df.columns


assert df.isnull().sum().sum() == 0

assert any(
    c.startswith("Location_")
    for c in df.columns
)
print("All check passed")

All check passed


In [27]:
df.isnull().sum()

Price              0
Odometer_km        0
Doors              0
Accidents          0
Year               0
Location_City      0
Location_Rural     0
Location_Suburb    0
CarAge             0
km_per_year        0
Km_per_year        0
Is_Urban           0
LogPrice           0
dtype: int64

In [28]:
#Statistics final

df.describe().round(2)

,Price,Odometer_km,Doors,Accidents,Year,Location_City,Location_Rural,Location_Suburb,CarAge,km_per_year,Km_per_year,Is_Urban,LogPrice
count,140.00,140.00,140.00,140.00,140.00,140.00,140.00,140.00,140.00,140.00,140.00,140.00,140.00
mean,3175.46,0.00,0.00,0.00,0.00,0.49,0.15,0.36,-0.00,0.00,0.00,0.49,7.80
std,2601.85,1.00,1.00,1.00,1.00,0.50,0.36,0.48,1.00,1.00,1.00,0.50,0.68
min,1500.00,-2.35,-2.12,-0.82,-1.69,0.00,0.00,0.00,-1.76,-1.02,-1.02,0.00,7.31
25%,1500.00,-0.62,-0.93,-0.82,-0.86,0.00,0.00,0.00,-0.83,-0.59,-0.59,0.00,7.31
50%,1500.00,-0.04,0.25,-0.82,-0.03,0.00,0.00,0.00,0.03,-0.28,-0.28,0.00,7.31
75%,4489.75,0.68,0.25,0.32,0.83,1.00,0.00,1.00,0.86,0.21,0.21,1.00,8.41
max,8974.38,2.63,1.44,2.59,1.76,1.00,1.00,1.00,1.69,5.40,5.40,1.00,9.10


In [29]:
# Save cleaned dataset

df.to_csv(
    "clean_car_dataset.csv",
    index=False
)

print("Saved.")

Saved.
